# MAGs taxonomic classification

Now that each MAG should correspond to an individual taxa, we will perform a taxonomic classification to see which species are present in our samples.

**All the following codes were run on Euler. So they cannot be run on Jupyterhub.**

## 1. Import Kraken2 Database

We start by importing the PlusPF database from Kraken2 as it contains all the domains we're interested in. The database is capped at 16GB, since the complete one requires too much computational capacity.

In [ ]:
mosh annotate build-kraken-db \
    --p-collection pluspf16 \
    --o-kraken2-db $data_dir/kraken2_db16 \
    --o-bracken-db $data_dir/bracken_db16 \

## 2. Classification using kraken2

We run Kraken2 classification so that it can compare the analyzed genomes to a reference.


In [ ]:
mosh annotate classify-kraken2 \
    --i-seqs $data_dir/mags_derep_all_domains.qza \
    --i-db $data_dir/kraken2_db16.qza \
    --p-threads 30 \
    --p-memory-mapping False \
    --o-reports $data_dir/kraken2-reports-mags_db16.qza \
    --o-outputs $data_dir/kraken2-hits-mags_db16.qza

### 2.1 Removing Problematic MAG

We have now two new artifacts: `FeatureData[Kraken2Report % Properties('mags')]` and `FeatureData[Kraken2Output % Properties('mags')]` that we could not directly run into a Qiime-like taxonomy file. There was one problematic MAG found (that was not classified at a kingdom level). So we removed it; the `remove-mags_1`file contains the feature ID of the problematic MAG.

In [ ]:
mosh annotate filter-kraken2-results \
  --i-reports $data_dir/kraken2-reports-mags_db16.qza \
  --i-outputs $data_dir/kraken2-hits-mags_db16.qza \
  --m-metadata-file $data_dir/remove-mags_1.tsv \
  --p-exclude-ids \
  --o-filtered-reports $data_dir/kraken2-reports-mags_db16_filtered-1.qza \
  --o-filtered-outputs $data_dir/kraken2-hits-mags_db16_filtered-1.qza \

## 3. More Qimme2-like taxonomy

In [ ]:
mosh annotate kraken2-to-mag-features \
    --i-reports $data_dir/kraken2-reports-mags_db16_filtered-1.qza \
    --i-outputs $data_dir/kraken2-hits-mags_db16_filtered-1.qza \
    --o-taxonomy $data_dir/mags-taxonomy-db16-filtered.qza\
    --verbose

### 3.1 Get Visualization

In [ ]:
qiime metadata tabulate \
  --m-input-file $data_dir/mags-taxonomy-db16-filtered.qza \
  --o-visualization $data_dir/mags-taxonomy-db16-filtered.qzv

## 4. Taxa Bar plot (original)

This is the bar plot using the taxonomic classification obtained with the previous steps. As there were a lot of unclassified MAGs, Milo provided another classification method detailed in step 5.

In [ ]:
! qiime metadata tabulate \
  --m-input-file $data_dir/mags-taxonomy.qza \
  --o-visualization $data_dir/mags-taxonomy.qzv

## 5. Other Taxonomic Classification (by Milo)

The previous command generated a bar plot with mostly unclassified MAGs. So we decided to also run the taxonomy classification directly on the pre-processed reads of the samples, to see if it would result in more classified MAGs.

### 5.1 Remove unclassified MAGs
We remove the unassigned MAGs, in order to have only the information on matched organisms. 

In [ ]:
qiime taxa filter-table \
    --i-table $data_dir/mags_derep_ft_merged_filtered.qza \
    --i-taxonomy $data_dir/kraken2-mags-taxonomy_db16_milo.qza \
    --p-exclude Unassigned \
    --o-filtered-table $data_dir/mags-table-filtered-milo-no-unassigned.qza

### 5.2 Taxa Bar Plot

In [ ]:
qiime taxa barplot \
    --i-table $data_dir/mags-table-filtered-milo-no-unassigned.qza \
    --i-taxonomy $data_dir/kraken2-mags-taxonomy_db16_milo.qza \
    --m-metadata-file $data_dir/updog_metadata.tsv \
    --o-visualization $data_dir/mags-taxa-bar-plot-milo-no-unclassified.qzv